# Certified drop-ins vs. Qiskit V2 primitives — on simulator and hardware

Qiskit's V2 primitives answer two different questions:

| Primitive | Returns | Knob you set | Statistical guarantee |
|---|---|---|---|
| `EstimatorV2` | expectation value ⟨O⟩ | `precision` (target std-error) | none — a 1σ target, not coverage |
| `SamplerV2` | raw bitstring counts | `shots` (fixed count) | none — just draws |

`qiskit-shot-planner` adds two **certified** drop-ins that keep the same call shape but
replace the fixed knob with an adaptive stopping rule based on the empirical-Bernstein
concentration inequality:

| Drop-in | Certifies | Guarantee |
|---|---|---|
| `AdaptiveEstimator` | ⟨O⟩ | `\|value − ⟨O⟩\| ≤ ε` w.p. `≥ 1 − δ` |
| `AdaptiveSampler` | `P(outcome ∈ S)` | `\|prob − P(S)\| ≤ ε` w.p. `≥ 1 − δ` |

A concentration inequality binds to a **bounded scalar**: `EstimatorV2` exposes one (⟨O⟩)
directly, so `AdaptiveEstimator` is a direct counterpart; `SamplerV2`'s raw draws are not a
scalar, so `AdaptiveSampler` names a bounded *functional* of the outcomes — a target-outcome
probability — and certifies that.

This notebook runs all four on the **same** workload, first on `AerSimulator` and then
through the exact code path used on **real IBM hardware**.

## 0. Setup — one circuit, one observable, one target outcome

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2, SamplerV2

from qamp_shotplanner import AdaptiveEstimator, AdaptiveSampler
from qamp_shotplanner.backends.samplers import backend_sampler

EPS, DELTA = 0.02, 0.01

# A 2-qubit state:  0.8525|00> + 0.5227|11>
qc = QuantumCircuit(2)
qc.ry(1.1, 0)
qc.cx(0, 1)

observable = SparsePauliOp.from_list([("ZZ", 1.0), ("XI", 0.5), ("II", -0.3)])
target = "11"  # the outcome whose probability we want to certify

# Exact references (statevector) to check every method against:
exact_ev = float(Statevector(qc).expectation_value(observable).real)
exact_p = float(Statevector(qc).probabilities_dict().get(target, 0.0))
print(f"exact  <O>        = {exact_ev:+.4f}")
print(f"exact  P('{target}')    = {exact_p:.4f}")

exact  <O>        = +0.7000
exact  P('11')    = 0.2732


## 1. `EstimatorV2` vs `AdaptiveEstimator` (AerSimulator)

`EstimatorV2` returns a value at a fixed *precision* — a standard-error target, with no
coverage guarantee. `AdaptiveEstimator` instead stops when the empirical-Bernstein radius
reaches ε, and reports a **certified** value plus the shots it actually cost.

In [2]:
# --- EstimatorV2: fixed precision, no coverage guarantee ---
ev2 = EstimatorV2()
v_ev2 = float(ev2.run([(qc, observable)], precision=0.02).result()[0].data.evs)
print(f"EstimatorV2 (precision=0.02): value = {v_ev2:+.4f}   "
      f"(1-sigma target; NOT a coverage guarantee)")

# --- AdaptiveEstimator: certified (eps, delta), default exact statevector sampler ---
r = AdaptiveEstimator(epsilon=EPS, delta=DELTA).run([(qc, observable)])[0]
print(f"AdaptiveEstimator           : value = {r.value:+.4f}   shots = {r.shots:,}   "
      f"certificate |value - <O>| <= {r.epsilon} w.p. >= {1 - r.delta:.2f}")
print(f"  |value - exact| = {abs(r.value - exact_ev):.4f}  "
      f"(within eps: {abs(r.value - exact_ev) <= r.epsilon})")
print(f"  note: the shot count is driven by the high-variance XI term (<XI> = 0, sigma^2 ~ 1)")

EstimatorV2 (precision=0.02): value = +0.6776   (1-sigma target; NOT a coverage guarantee)


AdaptiveEstimator           : value = +0.6992   shots = 278,173   certificate |value - <O>| <= 0.02 w.p. >= 0.99
  |value - exact| = 0.0008  (within eps: True)
  note: the shot count is driven by the high-variance XI term (<XI> = 0, sigma^2 ~ 1)


## 2. `SamplerV2` vs `AdaptiveSampler` (AerSimulator)

`SamplerV2` draws a fixed number of shots and returns raw counts — you compute `P(target)`
yourself, with no guarantee on its accuracy. `AdaptiveSampler` names that probability as the
bounded scalar, maps each shot to the indicator `1[outcome == target]`, and stops when it is
certified to ±ε.

In [3]:
# --- SamplerV2: fixed shots, raw counts -> compute P(target) by hand ---
qc_meas = qc.copy()
qc_meas.measure_all()
FIXED = 4096
counts = SamplerV2().run([(qc_meas,)], shots=FIXED).result()[0].join_data().get_counts()
p_sv2 = counts.get(target, 0) / FIXED
print(f"SamplerV2 ({FIXED} shots): P('{target}') = {p_sv2:.4f}   "
      f"(no guarantee on this estimate)")

# --- AdaptiveSampler: certified P(target) ---
rs = AdaptiveSampler(epsilon=EPS, delta=DELTA).run([(qc, target)])[0]
print(f"AdaptiveSampler          : P('{target}') = {rs.probability:.4f}   shots = {rs.shots:,}   "
      f"certificate |prob - P| <= {rs.epsilon} w.p. >= {1 - rs.delta:.2f}")
print(f"  |prob - exact| = {abs(rs.probability - exact_p):.4f}  "
      f"(within eps: {abs(rs.probability - exact_p) <= rs.epsilon})")

SamplerV2 (4096 shots): P('11') = 0.2747   (no guarantee on this estimate)
AdaptiveSampler          : P('11') = 0.2677   shots = 6,623   certificate |prob - P| <= 0.02 w.p. >= 0.99
  |prob - exact| = 0.0055  (within eps: True)


### Variance-adaptivity

The sampler's cost tracks the target's variance: a near-deterministic outcome is certified in
far fewer shots than one near `p = 1/2`. This is the whole point of an empirical-Bernstein
rule — it spends shots where the uncertainty actually is.

In [4]:
half = QuantumCircuit(1); half.ry(np.pi / 2, 0)   # P('1') = 0.50 (max variance)
low = QuantumCircuit(1);  low.ry(0.20, 0)          # P('1') ~ 0.01 (low variance)
s = AdaptiveSampler(epsilon=EPS, delta=DELTA)
print(f"P ~ 0.50 target: {s.run([(half, '1')])[0].shots:,} shots")
print(f"P ~ 0.01 target: {s.run([(low,  '1')])[0].shots:,} shots  (low variance -> stops early)")

P ~ 0.50 target: 6,623 shots
P ~ 0.01 target: 2,479 shots  (low variance -> stops early)


## 3. Same API through the real-backend code path

Both drop-ins take the measurement backend only through a `sampler_factory`. The library's
`backend_sampler` submits each adaptive batch as a job — via `SamplerV2` for real IBM
backends, or `backend.run` for `AerSimulator`. So the *exact* code path you use on hardware
runs unchanged on the simulator; only the `backend` object differs.

In [5]:
def make_factory(backend):
    """A sampler_factory that routes the adaptive loop through a real backend.

    Works for both AdaptiveEstimator and AdaptiveSampler: both hand the factory a
    (circuit, value_map, seed), where value_map sends a bitstring to a per-shot value
    (the +/-1 Pauli eigenvalue for the estimator, or the {0,1} indicator for the sampler).
    """
    def factory(circ, value_map, seed):
        measured = circ.copy()
        measured.measure_all()          # the factory owns the readout
        return backend_sampler(measured, value_map, backend, seed=seed)
    return factory

aer = AerSimulator()  # a real shot-based backend (stand-in for hardware)

r_est = AdaptiveEstimator(epsilon=EPS, delta=DELTA,
                          sampler_factory=make_factory(aer)).run([(qc, observable)])[0]
print(f"AdaptiveEstimator via backend: value = {r_est.value:+.4f}  shots = {r_est.shots:,}  "
      f"(within eps: {abs(r_est.value - exact_ev) <= EPS})")

r_smp = AdaptiveSampler(epsilon=EPS, delta=DELTA,
                        sampler_factory=make_factory(aer)).run([(qc, target)])[0]
print(f"AdaptiveSampler   via backend: P('{target}') = {r_smp.probability:.4f}  shots = {r_smp.shots:,}  "
      f"(within eps: {abs(r_smp.probability - exact_p) <= EPS})")

AdaptiveEstimator via backend: value = +0.6983  shots = 278,173  (within eps: True)


AdaptiveSampler   via backend: P('11') = 0.2781  shots = 6,623  (within eps: True)


## 4. Running on real IBM hardware

The only change is the `backend` object — swap `AerSimulator()` for an `IBMBackend`. The
cell below is guarded so the notebook never submits a real job by accident; set
`RUN_HARDWARE = True` (with credentials configured) to execute it. Each adaptive batch
becomes a queued `SamplerV2` job, and the job ids are recorded on the returned sampler for
provenance.

Note the honest scope: the certificate is on *sampling* error relative to the **device**
expectation, not on hardware bias relative to the ideal value — see the three-level error
hierarchy in the docs.

In [6]:
RUN_HARDWARE = False  # set True with IBM credentials to actually submit

if RUN_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False)
    print(f"using {backend.name}")

    r_est = AdaptiveEstimator(epsilon=EPS, delta=DELTA,
                              sampler_factory=make_factory(backend)).run([(qc, observable)])[0]
    print(f"AdaptiveEstimator on {backend.name}: value = {r_est.value:+.4f}  shots = {r_est.shots:,}")

    r_smp = AdaptiveSampler(epsilon=EPS, delta=DELTA,
                            sampler_factory=make_factory(backend)).run([(qc, target)])[0]
    print(f"AdaptiveSampler   on {backend.name}: P('{target}') = {r_smp.probability:.4f}  shots = {r_smp.shots:,}")
else:
    print("RUN_HARDWARE is False - skipping the live submission.")
    print("To run on hardware: configure QiskitRuntimeService credentials, set RUN_HARDWARE=True,")
    print("and re-run. Nothing else changes - make_factory(backend) is the only difference.")

RUN_HARDWARE is False - skipping the live submission.
To run on hardware: configure QiskitRuntimeService credentials, set RUN_HARDWARE=True,
and re-run. Nothing else changes - make_factory(backend) is the only difference.


## Summary

- **`EstimatorV2` / `SamplerV2`** give you a value or raw counts at a **fixed** cost, with
  **no coverage guarantee** on the result.
- **`AdaptiveEstimator` / `AdaptiveSampler`** keep the same call shape but stop **adaptively**
  and return a result **certified** to `(ε, δ)`, together with the shots that certificate cost
  — spending shots in proportion to the observable's (or outcome's) variance.
- The **same code runs on `AerSimulator` and real IBM hardware**; only the `backend` handed to
  `make_factory` changes.
- A concentration inequality certifies a **bounded scalar**, so the sampler certifies a
  target-outcome *probability*, not the full output distribution (that would scale with support
  size and forfeit the variance advantage).